# <span style="color:darkblue"> Lecture 10c: Analyzing Experiments </span>

<font size = "5">



# <span style="color:darkblue"> I. Import Libraries </span>


<font size = "5">

Install `maketables` in your notebook environment with `pip install maketables` if it is not already installed.

In [1]:
# The "pandas" library is used for processing datasets
# The "numpy" is for numeric observations and random numbers
# The "matplotlib.pyplot" library is for creating graphs

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# We will "alias" two sublibraries in "statsmodels"
# "statsmodels.formula.api" contains functions to estimate models
# "statsmodels.api" contains general-use statistical options

import statsmodels.api as sm
import statsmodels.formula.api as smf
import maketables as mt

<font size = "5">

Import data

In [2]:
dataset = pd.read_stata("data_raw/malawiexperiment.dta")

# <span style="color:darkblue"> II. Context </span>


<font size = "5">

Today we will review a paper by Rebecca Dizon-Ross published <br>
in the American Economic Review (2019).

- In this study, researchers partnered with local schools in Malawi <br>
- This study evaluated the impacts of information about children's <br>
 academic performance on parents subsequent investments in their <br>
  children's education.


https://www.povertyactionlab.org/evaluation/effects-student-performance-information-parental-decision-making-malawi?lang=fr

https://www.nber.org/papers/w24610

<font size = "5">

Intervention

- Parents in Malawi with low literacy levels had trouble interpreting <br>
school report cards.Many parents were unaware that their children were <br>
struggling with school.
- The intervention altered the way that the schools engaged with <br>
 parents. It had an impact on reducing the information gaps.


<font size = "5">

Experimental Design

- Students were assigned to randomly to treatment and control <br>
with 50% probability
- The random assignment was done at the household level

<font size = "5">

Findings


<img src="figures/treatmenteffects_dizonross.png" alt="drawing" width="650"/>



<font size = "5">

At baseline (before the intervention)

- The graph on the left shows parental beliefs at baseline
- Parents of low performing students thought that they were doing <br>
better than they were. Ideally it should  be along the 45-degree line
- Similar results for treated and control groups at baseline because <br>
of randomization

At endline (after the intervention)

- Treated parents had more accurate perceptions of their children's <br>
performance
- The treatment effects varied depending on the baseline test scores


# <span style="color:darkblue"> II. Basic Descriptive Analysis </span>


<font size = "5">

Total number of children

In [3]:
len(dataset)

5268

<font size = "5">

Total number of households

- 2 children per households

In [6]:
unique_ids = pd.unique(dataset["hhid"])
len(unique_ids)

2634

<font size = "5">

Calculate number of treated and control

In [7]:
table = pd.crosstab(index = dataset['treat'],columns = "count")
table

col_0,count
treat,
Control,2654
Treatment,2614


# <span style="color:darkblue"> III. Testing Covariate Balance </span>


<font size = "5">

Subset treated and control observations

In [8]:
dataset_treated = dataset.query('treat == "Treatment"')
dataset_control = dataset.query('treat == "Control"')

<font size = "5">

Socio-economic information can be collected at baseline <br>
(before the experiment) 

In [9]:
variables_scores      = ["ave"]
variables_respondent  = ["lit","primary_resp_fem","age_par1","farmer"]
variables_household   = ["tot_kids","one_par"]
variables_student     = ["std","age","female","attendance_sv"]

<font size = "5" >

Check that characteristics are similar between treated and control <br>
at baseline

In [10]:
# Compute mean and standard deviation for the treated group
display(dataset_treated[variables_respondent].describe().loc[['mean', 'std']])

# Compute mean and standard deviation for the control group
display(dataset_control[variables_respondent].describe().loc[['mean', 'std']])

,lit,primary_resp_fem,age_par1,farmer
mean,0.675613,0.758990,40.974080,0.460587
std,0.468235,0.427778,11.290328,0.498541


,lit,primary_resp_fem,age_par1,farmer
mean,0.667426,0.773926,40.645455,0.465544
std,0.471225,0.418366,10.638597,0.498907


<font size = "5">



<font size = "5" >

Conduct a formal test of whether the coefficients are similar

- We should expect the coefficient on the treatment variable <br>
to be non-significant

In [12]:
reg_model = smf.ols("lit ~ treat ", dataset)
results = reg_model.fit(cov_type= "HC1")

mt.ETable([results], model_heads=["Baseline balance"]).make(type="gt")

GT(_tbl_data=  __index_level_0__ __index_level_1__                      0
0              coef   treat=Treatment     0.008 <br> (0.013)
1              coef         Intercept  0.667*** <br> (0.009)
2             stats      Observations                  5,242
3             stats                R²               0.000076, _body=<great_tables._gt_data.Body object at 0x000001EB19CC0860>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x000001EB19CC1FD0>, _spanners=Spanners([SpannerInfo(spanner_id='lit', spanner_level=2, spanner_label=Html(text='lit'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='Baseline balance', spanner_level=1, spanner_label=Html(text='Baseline balance'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x000001EB19CC3AA0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x000001EB19CC1D30>, _source_notes=[Html(text='Significance levels: * p &lt; 0.1, ** p &lt; 0.05, *** p &lt; 0.01. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x000001EB19CC10D0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x000001EB19CC0F80>, <great_tables._gt_data.FormatInfo object at 0x000001EB19CC38F0>, <great_tables._gt_data.FormatInfo object at 0x000001EB19CC13A0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value

<font size = "5">

Try it yourself!

<font size = "3">

- Obtain summary statistics for the mean and standard deviation for <br>
the other sets of baseline variables


In [10]:
# Write your own code




<font size = "5">

Try it yourself!

<font size = "3">

- Write a loop that runs different regressions of baseline covariates <br>
on the treatment variable. This can help you automate the process of <br>
testing for covariate balance

In [11]:
# Write your own code




# <span style="color:darkblue"> IV. Calculating Average Treatment Effect </span>


- Make sure to use robust standard errors

<font size = "5">

Effect of treatment on endline beliefs

In [12]:
reg_model = smf.ols("u_ave ~ treat ", dataset)
results = reg_model.fit(cov_type= "HC1")

mt.ETable([results], model_heads=["Treatment effect"]).make(type="gt")

GT(_tbl_data=  __index_level_0__ __index_level_1__                       0
0              coef   treat=Treatment  -7.422*** <br> (0.499)
1              coef         Intercept  63.563*** <br> (0.343)
2             stats      Observations                   5,244
3             stats                R²                   0.041, _body=<great_tables._gt_data.Body object at 0x000002744CFA5C70>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x000002744E6FC530>, _spanners=Spanners([SpannerInfo(spanner_id='u_ave', spanner_level=2, spanner_label=Html(text='u_ave'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='Treatment effect', spanner_level=1, spanner_label=Html(text='Treatment effect'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x000002744E6FCE30>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x000002744E6FEBD0>, _source_notes=[Html(text='Significance levels: * p &lt; 0.1, ** p &lt; 0.05, *** p &lt; 0.01. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x000002744E6FF410>, _formats=[<great_tables._gt_data.FormatInfo object at 0x000002744E6FEB70>, <great_tables._gt_data.FormatInfo object at 0x000002744E6FDEB0>, <great_tables._gt_data.FormatInfo object at 0x000002744E6FF9B0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', ty

<font size = "5">

Add baseline covariates

In [13]:
reg_model = smf.ols("u_ave ~ treat + ave ", dataset)
results = reg_model.fit(cov_type= "HC1")

mt.ETable([results], model_heads=["Treatment + baseline control"]).make(type="gt")

GT(_tbl_data=  __index_level_0__ __index_level_1__                       0
0              coef   treat=Treatment  -7.056*** <br> (0.432)
1              coef               ave   0.508*** <br> (0.013)
2              coef         Intercept  39.655*** <br> (0.680)
3             stats      Observations                   5,244
4             stats                R²                   0.273, _body=<great_tables._gt_data.Body object at 0x000002744E708890>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x0000027450E17260>, _spanners=Spanners([SpannerInfo(spanner_id='u_ave', spanner_level=2, spanner_label=Html(text='u_ave'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='Treatment + baseline control', spanner_level=1, spanner_label=Html(text='Treatment + baseline control'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x0000027450E15A30>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x0000027450E15A60>, _source_notes=[Html(text='Significance levels: * p &lt; 0.1, ** p &lt; 0.05, *** p &lt; 0.01. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x0000027450E163C0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x0000027450E16690>, <great_tables._gt_data.FormatInfo object at 0x0000027450E15670>, <great_tables._gt_data.FormatInfo object at 0x0000027450E16C90>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='p

<font size = "5">

Test for heterogeneous effects


In [14]:
reg_model = smf.ols("u_ave ~ treat + ave + treat*ave", dataset)
results = reg_model.fit(cov_type= "HC1")

mt.ETable([results], model_heads=["Heterogeneous effect"]).make(type="gt")

GT(_tbl_data=  __index_level_0__      __index_level_1__                        0
0              coef        treat=Treatment  -25.998*** <br> (1.238)
1              coef                    ave    0.305*** <br> (0.018)
2              coef  treat=Treatment × ave    0.405*** <br> (0.024)
3              coef              Intercept   49.188*** <br> (0.884)
4             stats           Observations                    5,244
5             stats                     R²                    0.309, _body=<great_tables._gt_data.Body object at 0x0000027450E51430>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x0000027450E51C70>, _spanners=Spanners([SpannerInfo(spanner_id='u_ave', spanner_level=2, spanner_label=Html(text='u_ave'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='Heterogeneous effect', spanner_level=1, spanner_label=Html(text='Heterogeneous effect'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x0000027450E50AA0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x0000027450E50CE0>, _source_notes=[Html(text='Significance levels: * p &lt; 0.1, ** p &lt; 0.05, *** p &lt; 0.01. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x0000027450E50920>, _formats=[<great_tables._gt_data.FormatInfo object at 0x0000027450E50D70>, <great_tables._gt_data.FormatInfo object at 0x0000027450E50320>, <great_tables._gt_data.FormatInfo object at 0x0000027450E51910>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', 

<font size = "5">

Try it yourself!

Test for heterogeneous effects using other baseline covariates!

In [15]:
# Write your own code




